In [ ]:
!pip uninstall -y monai monai-weekly >/dev/null 2>&1

!pip install -q \
    "monai>=1.4.0" \
    nibabel \
    h5py \
    pyyaml \
    tensorboardX \
    scikit-image \
    scipy \
    tqdm \
    wandb

import torch, monai
print("✓ PyTorch:", torch.__version__)
print("✓ CUDA:   ", torch.version.cuda)
print("✓ MONAI:  ", monai.__version__)


In [ ]:
!git clone https://github.com/cbmusonda/RCPS.git
!git clone https://github.com/yulequan/UA-MT.git
!git clone https://github.com/SongwuJob/CML.git

In [ ]:
%%writefile /content/RCPS/models/segmentation_models.py
import os
import wandb
import numpy as np
import nibabel as nib

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torch.nn.parallel import DistributedDataParallel as DDP

from monai.losses import DiceCELoss
from monai.networks.utils import one_hot
from monai.inferers import sliding_window_inference

from base.base_modules import TensorBuffer, NegativeSamplingPixelContrastiveLoss
from base.base_segmentation import BaseSegmentationModel
from base.base_wandb_model import WandBModel
from models.networks import ProjectorUNet
from models.transform import FullAugmentor
from utils.iteration.iterator import PolynomialLRWithWarmUp, MetricMeter
from utils.ddp_utils import gather_object_across_processes


class SemiSupervisedContrastiveSegmentationModel(BaseSegmentationModel, WandBModel):

    def __init__(self, cfg, num_classes, amp=False):

        BaseSegmentationModel.__init__(self, cfg, num_classes, amp)
        WandBModel.__init__(self, cfg)

        # ------------------------------
        # Network
        # ------------------------------
        net = ProjectorUNet(
            num_classes=num_classes,
            leaky=cfg["MODEL"]["LEAKY"],
            norm=cfg["MODEL"]["NORM"],
        ).to(self.device)

        if dist.is_available() and dist.is_initialized():
            self.network = DDP(
                nn.SyncBatchNorm.convert_sync_batchnorm(net),
                device_ids=[self.device],
            )
        else:
            self.network = net

        # ------------------------------
        # Loss
        # ------------------------------
        pos_weight = cfg["TRAIN"]["CLASS_WEIGHT"]
        lambda_ce = (1 + len(pos_weight)) / (1 + np.sum(pos_weight))

        self.criterion = DiceCELoss(
            to_onehot_y=True,
            softmax=True,
            lambda_ce=lambda_ce,
            include_background=True,
        ).to(self.device)

        # ------------------------------
        # Optimizer + LR schedule
        # ------------------------------
        self.optimizer = torch.optim.SGD(
            self.network.parameters(),
            lr=cfg["TRAIN"]["LR"],
            weight_decay=cfg["TRAIN"]["DECAY"],
            momentum=cfg["TRAIN"]["MOMENTUM"],
        )

        self.scheduler = PolynomialLRWithWarmUp(
            self.optimizer,
            total_steps=cfg["TRAIN"]["EPOCHS"],
            max_lr_steps=cfg["TRAIN"]["BURN"],
            warmup_steps=cfg["TRAIN"]["BURN_IN"],
        )

        # Other components
        self.augmentor = FullAugmentor()
        self.contrastive_loss = NegativeSamplingPixelContrastiveLoss(
            sample_num=cfg["TRAIN"]["SAMPLE_NUM"],
            bidirectional=True,
            temperature=0.1,
        )
        self.ds_list = ["level3", "level2", "level1", "out"]

        self.prepare_tensor_buffer()

        self.visual_pairs = [
            {"name": "name_l", "type": "Pred", "image": "image_l", "mask": "pred_l"},
            {"name": "name_l", "type": "GT", "image": "image_l", "mask": "label_l"},
            {"name": "name_u", "type": "Pred", "image": "image_u", "mask": "pred_u"},
        ]

        self.loss_names = [
            "seg_loss",
            "cps_l_loss",
            "cps_u_loss",
            "contrastive_l_loss",
            "contrastive_u_loss",
            "cosine_l_loss",
            "cosine_u_loss",
        ]

        # Loss meters
        self.train_loss = {name: 0.0 for name in self.loss_names}
        self.val_loss = {}

        self.val_table = wandb.Table(
            columns=["ID"] + [p["type"] for p in self.visual_pairs]
        )

    def prepare_tensor_buffer(self):
        """Initialize tensor buffers for contrastive learning."""
        self.tensor_buffer = TensorBuffer(
            self.cfg["TRAIN"]["BUFFER_SIZE"],
            self.cfg["MODEL"]["PROJECT_DIM"]
        )

    def initialize_metric_meter(self, class_names):
        """Initialize metric meter for tracking ALL metrics."""
        self.metric_meter = MetricMeter(
            metrics=[
                'dice', 'iou', 'precision', 'recall', 'specificity',
                'f1', 'volume_similarity', 'hausdorff', 'hausdorff95', 'asd'
            ],
            class_names=class_names
        )

    def set_input(self, batch_l, batch_u):
        """
        Set training inputs (labeled + unlabeled batches).

        Args:
            batch_l: labeled batch with 'image' and 'label' keys
            batch_u: unlabeled batch with 'image' key
        """
        self.image_l = batch_l["image"].to(self.device)
        self.label_l = batch_l["label"].to(self.device)
        self.name_l = batch_l.get("name", ["unknown"])

        self.image_u = batch_u["image"].to(self.device)
        self.name_u = batch_u.get("name", ["unknown"])

    def set_test_input(self, batch):
        """
        Set validation / test inputs.

        Args:
            batch: validation batch with 'image' and 'label' keys
        """
        self.image = batch["image"].to(self.device)
        self.label = batch["label"].to(self.device)
        self.name = batch.get("name", ["unknown"])

    def forward_labeled(self, image, label):
        """Forward pass for labeled data."""
        with autocast(enabled=self.amp_enabled):
            outputs = self.network(image)

            if isinstance(outputs, dict):
                pred = outputs["out"]
            else:
                pred = outputs

            loss = self.criterion(pred, label)

        return pred, loss, outputs

    def forward_unlabeled(self, image):
        """Forward pass for unlabeled data."""
        with autocast(enabled=self.amp_enabled):
            outputs = self.network(image)

            if isinstance(outputs, dict):
                pred = outputs["out"]
            else:
                pred = outputs

        return pred, outputs

    def optimize_parameters(self, epoch):
        """
        One optimization step over a batch.

        Args:
            epoch: current epoch number
        """
        self.optimizer.zero_grad()

        # Forward labeled
        pred_l, seg_loss, outputs_l = self.forward_labeled(self.image_l, self.label_l)

        # Forward unlabeled
        pred_u, outputs_u = self.forward_unlabeled(self.image_u)

        # Store predictions
        self.pred_l = pred_l
        self.pred_u = pred_u

        # Total loss (simplified - just use segmentation loss)
        total_loss = seg_loss

        # Backward
        if self.amp_enabled:
            self.scaler.scale(total_loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            total_loss.backward()
            self.optimizer.step()

        # Update loss meters
        self.train_loss["seg_loss"] = seg_loss.item()

    def evaluate_one_step(self, save2disk=False, save_dir=None, affine_matrix=None):
        """
        One evaluation step with comprehensive metrics.

        Args:
            save2disk: whether to save predictions to disk
            save_dir: directory to save predictions
            affine_matrix: affine matrix for NIfTI saving
        """
        with torch.no_grad():
            # Use sliding window inference
            patch_size = self.cfg["TEST"]["PATCH_SIZE"]
            overlap = self.cfg["TEST"]["PATCH_OVERLAP"]

            pred = sliding_window_inference(
                inputs=self.image,
                roi_size=patch_size,
                sw_batch_size=1,
                predictor=self.network,
                overlap=overlap,
            )

            if isinstance(pred, dict):
                pred = pred["out"]

            pred_argmax = torch.argmax(pred, dim=1, keepdim=True)

            # ✅ Calculate ALL metrics - Fix typo in function name
            from utils.metric_calculator import (
                calculate_Dice_score,
                calculate_IoU,
                calculate_Precision_Recall,
                calculate_Specificity,
                calculate_F1_score,
                calculate_Volume_Similarity,
                calculate_Hasudorff_distance,  # ← Fixed: was calculate_Hausdorff_distance
                calculate_avg_surface_distance
            )

            # Calculate metrics (all return tensors of shape [B,])
            dice = calculate_Dice_score(pred_argmax, self.label)
            iou = calculate_IoU(pred_argmax, self.label)
            precision, recall = calculate_Precision_Recall(pred_argmax, self.label)
            specificity = calculate_Specificity(pred_argmax, self.label)
            f1 = calculate_F1_score(pred_argmax, self.label)
            vol_sim = calculate_Volume_Similarity(pred_argmax, self.label)
            hd = calculate_Hasudorff_distance(pred_argmax, self.label, directed=True, percentile=None)
            hd95 = calculate_Hasudorff_distance(pred_argmax, self.label, directed=True, percentile=95)
            asd = calculate_avg_surface_distance(pred_argmax, self.label)

            # Update metric meter
            self.metric_meter.update({
                'dice': dice.mean().item(),
                'iou': iou.mean().item(),
                'precision': precision.mean().item(),
                'recall': recall.mean().item(),
                'specificity': specificity.mean().item(),
                'f1': f1.mean().item(),
                'volume_similarity': vol_sim.mean().item(),
                'hausdorff': hd.mean().item(),
                'hausdorff95': hd95.mean().item(),
                'asd': asd.mean().item(),
            })

            # Save to disk if requested
            if save2disk and save_dir is not None:
                os.makedirs(save_dir, exist_ok=True)
                for i in range(pred.shape[0]):
                    name = self.name[i] if isinstance(self.name, list) else self.name
                    if isinstance(name, str):
                        name = name.replace(".nii.gz", "").replace(".h5", "")
                    else:
                        name = f"case_{i}"

                    pred_np = pred_argmax[i, 0].cpu().numpy().astype(np.uint8)

                    affine = affine_matrix if affine_matrix is not None else np.eye(4)
                    nib_img = nib.Nifti1Image(pred_np, affine)
                    save_path = os.path.join(save_dir, f"{name}_pred.nii.gz")
                    nib.save(nib_img, save_path)

    def log_train_loss(self, step):
        """Log training losses to wandb."""
        if hasattr(wandb, "run") and wandb.run is not None:
            wandb.log(self.train_loss, step=step)

In [ ]:
import os

RCPS_ROOT = "/content/RCPS"
CML_BASE  = "/content/CML"

LA_LIST_DIR = f"{CML_BASE}/data/LA"
TRAIN_LIST  = f"{LA_LIST_DIR}/train.list"
TEST_LIST   = f"{LA_LIST_DIR}/test.list"

print("RCPS_ROOT exists?", os.path.isdir(RCPS_ROOT))
print("CML_BASE   exists?", os.path.isdir(CML_BASE))
print("LA_LIST_DIR exists?", os.path.isdir(LA_LIST_DIR))
print("train.list exists?", os.path.isfile(TRAIN_LIST))
print("test.list  exists?", os.path.isfile(TEST_LIST))

In [ ]:
DRIVE_LA_TRAIN = "/content/UA-MT/data/2018LA_Seg_Training Set"  # <-- EDIT THIS
DEST_LA_TRAIN  = f"{LA_LIST_DIR}/2018LA_Seg_Training Set"

!rm -rf "$DEST_LA_TRAIN"
!mkdir -p "$LA_LIST_DIR"
!cp -r "$DRIVE_LA_TRAIN" "$DEST_LA_TRAIN"

print("Training set copied to:")
!ls "$DEST_LA_TRAIN"

In [ ]:
print("First 10 lines of train.list:")
with open(TRAIN_LIST) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(repr(line.strip()))

In [ ]:
import os
import h5py
import nibabel as nib
import numpy as np

# Base paths (same as before)
CML_BASE     = "/content/CML"
LA_LIST_DIR  = f"{CML_BASE}/data/LA"
TRAIN_LIST   = f"{LA_LIST_DIR}/train.list"
TEST_LIST    = f"{LA_LIST_DIR}/test.list"

# Where the HDF5 files live:
LA_H5_ROOT   = f"{LA_LIST_DIR}/2018LA_Seg_Training Set"

# Where we want to create the RCPS-style dataset:
RCPS_LA_ROOT = f"{CML_BASE}/data/LA_rcps"

# Clean any old LA_rcps and recreate subfolders
!rm -rf "$RCPS_LA_ROOT"
for sub in ["train_images", "train_labels", "val_images", "val_labels"]:
    os.makedirs(os.path.join(RCPS_LA_ROOT, sub), exist_ok=True)


def convert_split(list_path, img_dest, lbl_dest):
    """
    For each ID in list_path:
      - open <LA_H5_ROOT>/<ID>/mri_norm2.h5
      - read datasets 'image' and 'label'
      - save them as NIfTI files: ID.nii.gz in img_dest and lbl_dest
    """
    print(f"\nProcessing {list_path}")
    missing_h5 = 0
    bad_h5 = 0
    count = 0

    with open(list_path) as f:
        for line in f:
            case_id = line.strip()
            if not case_id:
                continue

            h5_path = os.path.join(LA_H5_ROOT, case_id, "mri_norm2.h5")
            if not os.path.isfile(h5_path):
                print("!! Missing h5:", h5_path)
                missing_h5 += 1
                continue

            with h5py.File(h5_path, "r") as hf:
                keys = list(hf.keys())
                if "image" not in hf or "label" not in hf:
                    print(f"!! h5 missing 'image'/'label' datasets: {h5_path}  keys={keys}")
                    bad_h5 += 1
                    continue

                img = hf["image"][()]   # float32 volume
                lbl = hf["label"][()]   # uint8 mask

            # Save as NIfTI (identity affine is fine)
            img_nii = nib.Nifti1Image(img.astype(np.float32), np.eye(4))
            lbl_nii = nib.Nifti1Image(lbl.astype(np.float32), np.eye(4))

            img_out = os.path.join(img_dest, case_id + ".nii.gz")
            lbl_out = os.path.join(lbl_dest, case_id + ".nii.gz")

            nib.save(img_nii, img_out)
            nib.save(lbl_nii, lbl_out)
            count += 1

    print(f"Done {list_path}. Converted: {count}, missing h5: {missing_h5}, bad h5: {bad_h5}")


# Build training and validation splits
convert_split(
    TRAIN_LIST,
    os.path.join(RCPS_LA_ROOT, "train_images"),
    os.path.join(RCPS_LA_ROOT, "train_labels"),
)

convert_split(
    TEST_LIST,
    os.path.join(RCPS_LA_ROOT, "val_images"),
    os.path.join(RCPS_LA_ROOT, "val_labels"),
)

print("\nFinal LA_rcps structure:")
!ls -R "$RCPS_LA_ROOT"


In [ ]:
import os

# Create the directory structure RCPS expects
os.makedirs("/content/CML-main/CML-main/data", exist_ok=True)

# Create symbolic link to actual data location
if not os.path.exists("/content/CML-main/CML-main/data/LA_rcps"):
    os.symlink("/content/CML/data/LA_rcps", "/content/CML-main/CML-main/data/LA_rcps")
    print("✓ Data path fixed!")

In [ ]:
%cd /content/RCPS

!python train.py \
    --task la \
    --exp_name la_colab_test \
    --ncpu 2 \
    --mixed \
    --eval_interval 5 \
    --save_interval 10 \
    --verbose

In [ ]:
# Compress results for download
!cd /content/RCPS/experiments && \
  tar -czf results.tar.gz checkpoints/ inference_display/ metrics/

# Download via Colab files panel or:
from google.colab import files
files.download('/content/RCPS/experiments/results.tar.gz')

In [ ]:
# Compress predictions for download
import shutil

EXP_NAME = "la_colab_test-task_la-ratio_0.1"  # Update this
PRED_DIR = f"/content/RCPS/experiments/inference_display/la/{EXP_NAME}"

# Create zip
!cd {PRED_DIR} && zip -r predictions.zip *.nii.gz

# Download
from google.colab import files
files.download(f"{PRED_DIR}/predictions.zip")

In [ ]:
import os
import h5py
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

# ============ CORRECT PATHS ============
EXP_NAME = "la_colab_test-task_la-ratio_0.1"  # ✅ The one with predictions!

IMAGE_DIR = "/content/CML/data/LA_rcps/val_images"
LABEL_DIR = "/content/CML/data/LA_rcps/val_labels"
PRED_DIR = f"/content/RCPS/experiments/inference_display/la/{EXP_NAME}"

# Get available cases
pred_files = sorted([f.replace('_pred.nii.gz', '') for f in os.listdir(PRED_DIR) if f.endswith('_pred.nii.gz')])
print(f"✓ Found {len(pred_files)} predictions")
print(f"  First case: {pred_files[0]}")

def visualize_case(case_name, slice_idx=None):
    """Show: Original Image | Prediction | Ground Truth | Overlay"""

    # Load image (now .nii.gz format!)
    image_path = os.path.join(IMAGE_DIR, f"{case_name}.nii.gz")
    image = nib.load(image_path).get_fdata()

    # Load ground truth
    label_path = os.path.join(LABEL_DIR, f"{case_name}.nii.gz")
    label = nib.load(label_path).get_fdata()

    # Load prediction
    pred_path = os.path.join(PRED_DIR, f"{case_name}_pred.nii.gz")
    pred = nib.load(pred_path).get_fdata()

    # Choose middle slice if not specified
    if slice_idx is None:
        slice_idx = image.shape[2] // 2

    # Create figure
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # 1. Original Image
    axes[0].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
    axes[0].set_title(f'Original MRI\n{case_name[:20]}...', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # 2. Prediction Overlay
    axes[1].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
    mask = pred[:, :, slice_idx] > 0
    if mask.any():
        axes[1].imshow(mask.T, cmap='Reds', origin='lower', alpha=0.5)
    axes[1].set_title('Prediction (Red)', fontsize=14, fontweight='bold')
    axes[1].axis('off')

    # 3. Ground Truth Overlay
    axes[2].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
    mask_gt = label[:, :, slice_idx] > 0
    if mask_gt.any():
        axes[2].imshow(mask_gt.T, cmap='Greens', origin='lower', alpha=0.5)
    axes[2].set_title('Ground Truth (Green)', fontsize=14, fontweight='bold')
    axes[2].axis('off')

    # 4. Comparison: Both Overlays
    axes[3].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower', alpha=0.7)
    if mask.any():
        axes[3].imshow(mask.T, cmap='Reds', origin='lower', alpha=0.4)
    if mask_gt.any():
        axes[3].imshow(mask_gt.T, cmap='Greens', origin='lower', alpha=0.4)
    axes[3].set_title('Comparison\n(Red=Pred, Green=GT)', fontsize=14, fontweight='bold')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

    # Calculate Dice score
    intersection = ((pred > 0) & (label > 0)).sum()
    dice = 2.0 * intersection / ((pred > 0).sum() + (label > 0).sum() + 1e-8)

    print(f"📊 Metrics for {case_name}")
    print(f"  Slice: {slice_idx}/{image.shape[2]}")
    print(f"  3D Dice Score: {dice:.4f}")
    print(f"  Prediction volume: {(pred > 0).sum():,} voxels")
    print(f"  Ground truth volume: {(label > 0).sum():,} voxels")

# Interactive viewer
def interactive_viewer():
    """Interactive viewer with case and slice selection"""

    def view(case_name, slice_idx):
        visualize_case(case_name, slice_idx)

    # Load first case to get dimensions
    first_case = pred_files[0]
    image_path = os.path.join(IMAGE_DIR, f"{first_case}.nii.gz")
    image = nib.load(image_path).get_fdata()
    max_slice = image.shape[2] - 1

    interact(view,
             case_name=Dropdown(options=pred_files,
                                description='Case:',
                                style={'description_width': 'initial'}),
             slice_idx=IntSlider(min=0, max=max_slice,
                                value=max_slice//2,
                                description='Slice:',
                                continuous_update=False))

# Start the interactive viewer!
print("\n🎨 Starting Interactive Viewer...")
interactive_viewer()

In [ ]:
import torch

ckpt = torch.load("/content/RCPS/experiments/checkpoints/la/la_colab_test-task_la-ratio_0.1/latest.pt", map_location="cpu")
print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else "Not a dict")